In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import urllib.request
import json
import time
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm

# ==========================================
# 1. 基础配置 (Configuration)
# ==========================================

class Config:
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0

        # 进一步优化的超参数
        self.embed_dim = 64   # 恢复到64以容纳更复杂的特征
        self.seq_len = 100
        self.batch_size = 32
        self.epochs = 50      # 增加训练轮数
        self.lr = 0.001       # 降低学习率，配合 Scheduler
        self.dropout = 0.45   # 略微提高 Dropout
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        self.data_dir = "./data/math2015"
        self.dataset_name = "math2015"
        self.llm_sample_size = 5

# ==========================================
# 2. 数据处理 (Data Processing)
# ==========================================

class Math2015Processor:
    def __init__(self, config):
        self.config = config
        if not os.path.exists(config.data_dir):
            os.makedirs(config.data_dir)

    def _find_files(self, path):
        data_file, q_file = None, None
        for root, _, files in os.walk(path):
            for file in files:
                low_name = file.lower()
                if any(k in low_name for k in ["data", "mat"]) and file.endswith((".txt", ".csv")):
                    if "termsofuse" not in low_name:
                        data_file = os.path.join(root, file)
                if low_name.startswith("q") and file.endswith((".txt", ".csv")):
                    q_file = os.path.join(root, file)
        return data_file, q_file

    def process(self):
        data_path, q_path = self._find_files(self.config.data_dir)
        if not data_path:
            print("正在下载数据集...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])
                from EduData import get_data
                get_data(self.config.dataset_name, self.config.data_dir)
            except: pass
            data_path, q_path = self._find_files(self.config.data_dir)

        if not data_path: raise FileNotFoundError("未发现数据文件。")

        df_raw = pd.read_csv(data_path, sep=None, engine='python', header=None).dropna(axis=1, how='all')
        df_raw = df_raw.apply(pd.to_numeric, errors='coerce').fillna(0)

        if df_raw.shape[1] > 5:
            df = df_raw.stack().reset_index()
            df.columns = ['user_id', 'item_id', 'score']
        else:
            df = df_raw.copy()
            df.columns = ['user_id', 'item_id', 'score'] + list(df.columns[3:])

        q_df = pd.read_csv(q_path, sep=None, engine='python', header=None).dropna(axis=1, how='all')
        q_matrix_raw = q_df.apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(float)

        max_item_id = int(df['item_id'].max())
        if q_matrix_raw.shape[0] <= max_item_id:
            padding_rows = max_item_id - q_matrix_raw.shape[0] + 1
            q_matrix = np.vstack([q_matrix_raw, np.zeros((padding_rows, q_matrix_raw.shape[1]))])
        else:
            q_matrix = q_matrix_raw

        self.config.num_questions, self.config.num_concepts = q_matrix.shape
        self.config.num_students = int(df['user_id'].max() + 1)

        df['uid_idx'] = df['user_id'].astype(int) + 1
        df['iid_idx'] = df['item_id'].astype(int) + 1
        df['score'] = df['score'].astype(float)

        concept_adj = np.dot(q_matrix.T, q_matrix)
        concept_adj = (concept_adj > 0).astype(float)

        sequences = []
        for uid, group in tqdm(df.groupby('uid_idx'), desc="生成序列数据"):
            if len(group) < 3: continue
            sequences.append({
                'uid': int(uid),
                'iids': [int(i) for i in group['iid_idx'].values],
                'labels': [float(l) for l in group['score'].values]
            })

        records = df[['uid_idx', 'iid_idx', 'score']].values.tolist()
        return sequences, records, q_matrix, concept_adj

# ==========================================
# 3. 极速提升版的 HIG-TCAN_CD 模型
# ==========================================

class HIG_TCAN_CD_v2(nn.Module):
    def __init__(self, config, q_matrix):
        super().__init__()
        self.embed_dim = config.embed_dim

        # 1. 异构嵌入层
        self.q_emb = nn.Embedding(config.num_questions + 2, config.embed_dim, padding_idx=0)
        self.k_emb = nn.Embedding(config.num_concepts + 2, config.embed_dim, padding_idx=0)
        self.s_emb = nn.Embedding(config.num_students + 2, config.embed_dim, padding_idx=0)

        # 2. 交互嵌入层 (核心增强)
        # 将 (题目, 结果) 编码，结果 0 或 1。由于是连续值，我们用线性映射代替简单的 Embedding
        self.interaction_proj = nn.Linear(config.embed_dim + 1, config.embed_dim)

        self.register_buffer('q_k_rel', torch.tensor(q_matrix, dtype=torch.float32))

        # 3. 时间卷积层 (TCN 增强)
        self.conv1 = nn.Conv1d(config.embed_dim, config.embed_dim, kernel_size=3, padding=2, dilation=1)
        self.conv2 = nn.Conv1d(config.embed_dim, config.embed_dim, kernel_size=3, padding=4, dilation=2)

        # 4. 注意力层
        self.query = nn.Linear(config.embed_dim, config.embed_dim)
        self.key = nn.Linear(config.embed_dim, config.embed_dim)
        self.value = nn.Linear(config.embed_dim, config.embed_dim)

        self.layer_norm = nn.LayerNorm(config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # 5. 预测层
        self.pred_mlp = nn.Sequential(
            nn.Linear(config.embed_dim * 3, 128),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, q_ids, labels, u_id):
        # 创建 Mask
        mask = (q_ids > 0).float()

        # A. 异构聚合获取题目特征
        num_k = self.q_k_rel.size(1)
        k_feat_base = torch.matmul(self.q_k_rel, self.k_emb.weight[1:num_k+1])
        k_feat_all = torch.cat([torch.zeros(1, self.embed_dim, device=q_ids.device), k_feat_base], dim=0)
        q_feat = self.q_emb(q_ids) + F.embedding(q_ids, k_feat_all)
        q_feat = q_feat * mask.unsqueeze(-1)

        # B. 构造交互序列 (iid + score)
        interaction_input = torch.cat([q_feat, labels.unsqueeze(-1)], dim=-1)
        x = self.interaction_proj(interaction_input)

        # C. TCN 特征提取 (局部模式)
        x_conv = x.permute(0, 2, 1)
        x_conv = F.relu(self.conv1(x_conv))[:, :, :q_ids.size(1)]
        x_conv = F.relu(self.conv2(x_conv))[:, :, :q_ids.size(1)]
        x = x_conv.permute(0, 2, 1) + x # 残差

        # D. Temporal Attention (全局依赖)
        B, L, D = x.size()
        Q, K, V = self.query(x), self.key(x), self.value(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)

        causal_mask = torch.tril(torch.ones(L, L, device=x.device)).view(1, L, L)
        scores = scores.masked_fill(causal_mask == 0, -1e9)
        scores = scores.masked_fill(mask.unsqueeze(1) == 0, -1e9)

        attn = F.softmax(scores, dim=-1)
        h = torch.matmul(self.dropout(attn), V)
        h = self.layer_norm(h + x)

        # E. 预测对齐
        s_static = self.s_emb(u_id).unsqueeze(1).expand(-1, L, -1)

        # 使用 t 的历史状态和学生能力，预测 t+1 题
        h_history = h[:, :-1, :]
        q_next = q_feat[:, 1:, :]
        s_current = s_static[:, 1:, :]

        feat = torch.cat([h_history, q_next, s_current], dim=-1)
        return self.pred_mlp(feat).squeeze(-1)

# ==========================================
# 4. 其他模型定义 (略，保持同上版本)
# ==========================================

class DINA(nn.Module):
    def __init__(self, n_user, n_item, n_skill, q_matrix):
        super().__init__()
        self.register_buffer('q_matrix', torch.tensor(q_matrix, dtype=torch.float32))
        self.mastery = nn.Embedding(n_user + 5, n_skill)
        self.slip = nn.Embedding(n_item + 5, 1)
        self.guess = nn.Embedding(n_item + 5, 1)
    def forward(self, uid, iid):
        m = torch.sigmoid(self.mastery(uid))
        q_idx = (iid - 1).clamp(0, self.q_matrix.size(0) - 1)
        q = self.q_matrix[q_idx]
        eta = torch.prod(torch.pow(m, q), dim=-1, keepdim=True)
        s, g = torch.sigmoid(self.slip(iid)), torch.sigmoid(self.guess(iid))
        return (torch.pow(1-s, eta) * torch.pow(g, 1-eta)).squeeze()

class GCCD(nn.Module):
    def __init__(self, n_user, n_item, n_skill, q_matrix):
        super().__init__()
        self.u_emb = nn.Embedding(n_user + 5, n_skill)
        self.i_emb = nn.Embedding(n_item + 5, n_skill)
        self.register_buffer('q_matrix', torch.tensor(q_matrix, dtype=torch.float32))
        self.fc = nn.Sequential(nn.Linear(n_skill, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, uid, iid):
        u, i = torch.sigmoid(self.u_emb(uid)), torch.sigmoid(self.i_emb(iid))
        q_idx = (iid - 1).clamp(0, self.q_matrix.size(0) - 1)
        q = self.q_matrix[q_idx]
        return self.fc((u - i) * q).squeeze()

class GCDM(nn.Module):
    def __init__(self, n_user, n_item, n_skill, q_matrix, concept_adj):
        super().__init__()
        self.u_emb = nn.Embedding(n_user + 5, n_skill)
        self.i_emb = nn.Embedding(n_item + 5, n_skill)
        self.register_buffer('q_matrix', torch.tensor(q_matrix, dtype=torch.float32))
        self.register_buffer('adj', torch.tensor(concept_adj, dtype=torch.float32))
        self.fc = nn.Sequential(nn.Linear(n_skill, 1), nn.Sigmoid())
    def forward(self, uid, iid):
        u = torch.sigmoid(self.u_emb(uid))
        u_smooth = torch.matmul(u, self.adj) / (self.adj.sum(dim=-1) + 1e-8)
        i = torch.sigmoid(self.i_emb(iid))
        q_idx = (iid - 1).clamp(0, self.q_matrix.size(0) - 1)
        return self.fc((u_smooth - i) * self.q_matrix[q_idx]).squeeze()

# ==========================================
# 5. 实验运行器
# ==========================================

class Runner:
    def __init__(self, config):
        self.config = config
        self.results = {}

    def train(self, name, model, train_loader, test_loader, is_seq=False):
        model.to(self.config.device)
        optimizer = torch.optim.Adam(model.parameters(), lr=self.config.lr, weight_decay=1e-4)
        # 引入指数衰减
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.config.epochs)

        print(f"\n>>> 训练优化版 v2: {name}")
        best_auc = 0
        for epoch in range(self.config.epochs):
            model.train()
            for batch in train_loader:
                optimizer.zero_grad()
                if is_seq:
                    q, r, u = [b.to(self.config.device) for b in batch]
                    mask = (q[:, 1:] > 0).float()
                    preds = model(q, r, u)
                    targets = r[:, 1:]
                    loss = F.binary_cross_entropy(preds, targets, reduction='none')
                    loss = (loss * mask).sum() / (mask.sum() + 1e-8)
                else:
                    u, i, r = [b.to(self.config.device) for b in batch]
                    loss = F.binary_cross_entropy(model(u.long(), i.long()), r.float())

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5) # 调低裁剪阈值
                optimizer.step()
            scheduler.step()

        # 评估
        model.eval(); all_p, all_r = [], []
        with torch.no_grad():
            for batch in test_loader:
                if is_seq:
                    q, r, u = [b.to(self.config.device) for b in batch]
                    mask = (q[:, 1:] > 0)
                    preds = model(q, r, u).cpu().numpy()[mask.cpu().numpy()]
                    targets = r[:, 1:].cpu().numpy()[mask.cpu().numpy()]
                    all_p.extend(preds); all_r.extend(targets)
                else:
                    u, i, r = [b.to(self.config.device) for b in batch]
                    all_p.extend(model(u.long(), i.long()).cpu().numpy())
                    all_r.extend(r.cpu().numpy())

        y_true = (np.array(all_r) >= 0.5).astype(int)
        y_pred = np.array(all_p)
        auc = roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.5
        acc = accuracy_score(y_true, (y_pred > 0.5).astype(int))
        self.results[name] = {'AUC': auc, 'ACC': acc}
        print(f"{name} 完成. AUC: {auc:.4f} | ACC: {acc:.4f}")

def main():
    config = Config()
    proc = Math2015Processor(config)
    seqs, recs, q_mat, concept_adj = proc.process()
    runner = Runner(config)

    train_r, test_r = train_test_split(recs, test_size=0.2, random_state=42)
    static_tr = DataLoader(train_r, batch_size=config.batch_size, shuffle=True)
    static_te = DataLoader(test_r, batch_size=config.batch_size)

    def collate_seq(batch):
        qs = torch.nn.utils.rnn.pad_sequence([torch.tensor(x['iids'][:config.seq_len]) for x in batch], batch_first=True, padding_value=0)
        rs = torch.nn.utils.rnn.pad_sequence([torch.tensor(x['labels'][:config.seq_len]) for x in batch], batch_first=True, padding_value=0).float()
        us = torch.tensor([x['uid'] for x in batch])
        return qs, rs, us

    train_s, test_s = train_test_split(seqs, test_size=0.2, random_state=42)
    seq_tr = DataLoader(train_s, batch_size=config.batch_size, collate_fn=collate_seq, shuffle=True)
    seq_te = DataLoader(test_s, batch_size=config.batch_size, collate_fn=collate_seq)

    # 对比实验
    runner.train("DINA", DINA(config.num_students, config.num_questions, config.num_concepts, q_mat), static_tr, static_te)
    runner.train("GCCD", GCCD(config.num_students, config.num_questions, config.num_concepts, q_mat), static_tr, static_te)
    runner.train("GCDM", GCDM(config.num_students, config.num_questions, config.num_concepts, q_mat, concept_adj), static_tr, static_te)
    # 训练极速提升版的 HIG-TCAN-CD v2
    runner.train("HIG-TCAN-CD", HIG_TCAN_CD_v2(config, q_mat), seq_tr, seq_te, is_seq=True)

    print("\n" + "="*45)
    print(f"{'Model':<18} | {'AUC':<10} | {'ACC':<10}")
    print("-" * 45)
    for model, m in runner.results.items():
        print(f"{model:<18} | {m['AUC']:.4f}     | {m['ACC']:.4f}")
    print("="*45)

if __name__ == "__main__":
    main()

正在下载数据集...


downloader, INFO http://staff.ustc.edu.cn/~qiliuql/data/math2015.rar is saved as data/math2015/math2015.rar


downloader, INFO data/math2015/math2015.rar is unrar to data/math2015/math2015


生成序列数据: 100%|██████████| 3911/3911 [00:00<00:00, 10026.60it/s]



>>> 训练优化版 v2: DINA
DINA 完成. AUC: 0.7468 | ACC: 0.6876

>>> 训练优化版 v2: GCCD
GCCD 完成. AUC: 0.6848 | ACC: 0.6462

>>> 训练优化版 v2: GCDM
GCDM 完成. AUC: 0.7182 | ACC: 0.6717

>>> 训练优化版 v2: HIG-TCAN-CD
HIG-TCAN-CD 完成. AUC: 0.7690 | ACC: 0.7001

Model              | AUC        | ACC       
---------------------------------------------
DINA               | 0.7468     | 0.6876
GCCD               | 0.6848     | 0.6462
GCDM               | 0.7182     | 0.6717
HIG-TCAN-CD        | 0.7690     | 0.7001
